In [0]:
%python
import dlt
import re
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
import builtins

TAXI_SOURCE_TABLE = "/Volumes/students_data/team-1-data-schema/team-1-data-schema-volume/taxi_data.csv"

In [0]:
%sql
USE CATALOG `students_data`;
USE SCHEMA `team-1-data-schema`;

## Silver Layer — `silver_taxi_data`

This table reads from `bronze_taxi_data` and applies the following transformations:

### 1. Deduplication
Window function (`ROW_NUMBER` partitioned by `booking_id`, ordered by `ingestion_timestamp DESC`) keeps only the latest record per natural key — defensive against duplicate ingestion.

### 2. Type casting & standardisation
* All columns cast from string to correct types (timestamps, doubles, integers)
* `driver`: `#` prefix stripped, cast to int
* `price`: commas removed, cast to double
* `capabilities`: code letters mapped to readable names via UDF (e.g. `ZM6` → `Card Reader, Minibus, 6 seater`)
* `payment_type`: overridden to `Card` when vehicle has card reader capability (`Z`)
* `booked_by`: nulls defaulted to `booking_source`

### 3. Filtering
* Completed trips with `price < 3.20` removed; cancelled/no-fare preserved for gold analysis
* Destination coordinates must fall within Ireland/UK bounding box (lat 51.4–55.5, lon -10.8 to -5.2)

### 4. Entity conformity
* Zone codes (`pickup_zone`, `destination_zone`) verified: no whitespace, case, or spelling inconsistencies across 49 zones. Shared vocabulary between pickup and destination confirmed.

### 5. Derived columns & flags - NOTE: Should reside on the Gold layer rather than Silver
* `trip_duration_minutes`, `total_time_taken`, `wait_time_minutes`, `price_per_mile`
* `is_completed`, `is_valid_distance`, `is_valid_timing`, `is_valid_duration`, `is_price_outlier`

### 6. Data quality expectations (DLT)
Seven `@dlt.expect` rules track quality metrics without dropping rows — visible in the pipeline UI.

In [0]:
%python
# ---------------------------------------------------------------------------
# Capability transformation UDF
# ---------------------------------------------------------------------------
@udf(returnType=StringType())
def transform_capabilities(cap):
    """
    Transforms capability codes into readable names.
    - Removes 'HL'/'LH' pairs, keeping remaining characters
    - Maps known letters to names (Z=Card Reader, D=Delivery, etc.)
    - Unmapped letters → single 'Other'
    - Multiple digits → keep highest; digits below 6 are stripped
    """
    if cap is None or cap.strip() == "":
        return None

    letter_map = {
        "Z": "Card Reader", "D": "Delivery", "H": "High Car", "L": "Low Car",
        "W": "Wheelchair", "M": "Minibus", "F": "Female", "V": "VIP",
        "T": "Tour", "P": "Pet",
    }
    number_map = {6: "6 seater", 7: "7 seater", 8: "8 seater"}

    # Step 1 – Remove HL / LH pairs
    cap = cap.replace("HL", "").replace("LH", "")
    if not cap.strip():
        return None

    results = []
    has_other = False
    numbers = []

    for char in cap:
        if char.isdigit():
            numbers.append(int(char))
        elif char.isalpha():
            mapped = letter_map.get(char.upper())
            if mapped:
                results.append(mapped)
            else:
                has_other = True

    # Step 2 – Numbers: keep only the highest digit; include only if >= 6
    if numbers:
        max_num = builtins.max(numbers)
        if max_num in number_map:
            results.append(number_map[max_num])
        elif max_num >= 6:
            has_other = True
        # digits below 6 are stripped entirely

    if has_other:
        results.append("Other")

    return ", ".join(results) if results else None


# ---------------------------------------------------------------------------
# Silver DLT table definition
# ---------------------------------------------------------------------------
# IQR-based price outlier threshold: Q3 + 3*IQR = 8.16 per mile
# Derived from analysis of completed trips with distance > 0
_PRICE_PER_MILE_UPPER_FENCE = 8.16


@dlt.table(
    name="silver_taxi_data",
    comment="Cleaned taxi data with standardised types, flags, derived columns, and capability mappings"
)
@dlt.expect_or_drop("valid_booking_id", "booking_id IS NOT NULL")
@dlt.expect("valid_pickup_due", "pickup_due IS NOT NULL")
@dlt.expect("valid_coordinates", "pickup_latitude IS NOT NULL AND pickup_longitude IS NOT NULL")
@dlt.expect("non_negative_distance", "distance >= 0 OR distance IS NULL")
@dlt.expect("completed_has_arrival_time", "trip_status != 'Completed' OR time_vehicle_arrived IS NOT NULL")
@dlt.expect("completed_has_pickup_time", "trip_status != 'Completed' OR time_picked_up IS NOT NULL")
@dlt.expect("valid_trip_duration", "trip_status != 'Completed' OR trip_duration_minutes >= 0")
def silver_taxi_data():
    df = dlt.read("bronze_taxi_data")

    # --- Deduplicate to correct grain ---
    # Keep only the latest record per booking_id (by ingestion_timestamp)
    # using a window function, defensive against duplicate ingestion runs
    dedup_window = Window.partitionBy("booking_id").orderBy(col("ingestion_timestamp").desc())
    df = df.withColumn("_row_num", row_number().over(dedup_window))
    df = df.filter(col("_row_num") == 1).drop("_row_num")

    # --- Rename source → trip_status ---
    df = df.withColumnRenamed("source", "trip_status")

    # --- Clean and cast all columns to proper types ---
    df = df.withColumns({
        "driver":                regexp_replace(col("driver"), "^#", "").cast("int"),
        "price":                 regexp_replace(col("price"), ",", "").cast("double"),
        "vehicle":               col("vehicle").cast("int"),
        "distance":              col("distance").cast("double"),
        "priority":              col("priority").cast("int"),
        "booking_id":            col("booking_id").cast("long"),
        "pickup_latitude":       col("pickup_latitude").cast("double"),
        "pickup_longitude":      col("pickup_longitude").cast("double"),
        "destination_latitude":  col("destination_latitude").cast("double"),
        "destination_longitude": col("destination_longitude").cast("double"),
        "pickup_due":            to_timestamp(col("pickup_due"), "dd/MM/yyyy HH:mm"),
        "completed":             to_timestamp(col("completed"), "dd/MM/yyyy HH:mm"),
        "time_dispatched":       to_timestamp(col("time_dispatched"), "dd/MM/yyyy HH:mm"),
        "time_vehicle_arrived":  to_timestamp(col("time_vehicle_arrived"), "dd/MM/yyyy HH:mm"),
        "time_picked_up":        to_timestamp(col("time_picked_up"), "dd/MM/yyyy HH:mm"),
    })

    # --- Hard filters ---
    # Minimum fare threshold applies only to completed trips;
    # cancelled / no-fare records are preserved for gold-layer analysis
    df = df.filter(
        (col("trip_status") != "Completed") | (col("price") >= 3.20)
    )

    # Destination coordinates must fall within Ireland/UK bounding box
    # Note: pickup/destination zone codes verified as consistent across 49 zones
    # (no whitespace, case, or spelling inconsistencies found)
    df = df.filter(
        (col("destination_latitude").between(51.4, 55.5))
        & (col("destination_longitude").between(-10.8, -5.2))
    )

    # --- Flags, defaults, and derived columns ---
    df = df.withColumns({
        "is_completed":       (col("trip_status") == "Completed").cast("boolean"),
        "is_valid_distance":  (col("distance") > 0).cast("boolean"),
        "is_valid_timing":    (
            col("time_vehicle_arrived").isNotNull() & col("time_picked_up").isNotNull()
        ).cast("boolean"),
        "is_valid_duration":  (
            col("trip_status") != "Completed"
        ).cast("boolean") | (
            (unix_timestamp(col("completed")) - unix_timestamp(col("pickup_due"))) >= 0
        ),
        "booked_by":          coalesce(col("booked_by"), col("booking_source")),
        "payment_type":       when(
            col("capabilities").contains("Z"), lit("Card")
        ).otherwise(col("payment_type")),
        "trip_duration_minutes": round(
            (unix_timestamp(col("completed")) - unix_timestamp(col("pickup_due"))) / 60, 2
        ),
        "total_time_taken": round(
            (unix_timestamp(col("completed")) - unix_timestamp(col("time_dispatched"))) / 60, 2
        ),
        "wait_time_minutes": round(
            (unix_timestamp(col("time_picked_up")) - unix_timestamp(col("time_vehicle_arrived"))) / 60, 2
        ),
        "price_per_mile": when(
            col("distance") > 0,
            round(col("price") / col("distance"), 2)
        ).otherwise(lit(None)),
        "is_price_outlier": when(
            (col("distance") > 0) & ((col("price") / col("distance")) > _PRICE_PER_MILE_UPPER_FENCE),
            lit(True)
        ).otherwise(lit(False)),
    })

    # --- Transform capability codes to readable names ---
    df = df.withColumns({
        "capabilities": transform_capabilities(col("capabilities")),
    })

    return df
       